In [3]:
import rasterio

with rasterio.open("DEM/37601/37601.img") as src:
    print(src.count)   # 밴드 수
    print(src.dtypes)  # 데이터 타입
    print(src.crs)     # 좌표계

1
('float32',)
EPSG:5179


In [4]:
with rasterio.open("DEM/37601/37601.img") as src:
    val = list(src.sample([(127.0, 37.5)]))[0][0]
    print(val)

-9999.0


In [6]:
import rasterio
from pyproj import Transformer

lon, lat = 127.0, 37.5

# WGS84 -> EPSG:5179
transformer = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
x_5179, y_5179 = transformer.transform(lon, lat)

with rasterio.open("DEM/37601/37601.img") as src:
    val = list(src.sample([(x_5179, y_5179)]))[0][0]
    print("변환 좌표:", x_5179, y_5179)
    print("고도값:", val)
    print("nodata:", src.nodata)

변환 좌표: 955804.8163413828 1944643.7157885316
고도값: -9999.0
nodata: -9999.0


In [7]:
with rasterio.open("DEM/37601/37601.img") as src:
    print(src.bounds)
    print(src.nodata)

BoundingBox(left=867690.0, bottom=1972350.0, right=890640.0, top=2000970.0)
-9999.0


In [9]:
from pathlib import Path
import rasterio
from pyproj import Transformer

# WGS84(경도,위도) -> EPSG:5179
transformer = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)

def find_elevation_from_tiles(lon, lat, dem_root="DEM"):
    x, y = transformer.transform(lon, lat)

    dem_files = list(Path(dem_root).rglob("*.img"))

    for dem_file in dem_files:
        with rasterio.open(dem_file) as src:
            b = src.bounds

            # 해당 타일 범위 안인지 확인
            if b.left <= x <= b.right and b.bottom <= y <= b.top:
                val = list(src.sample([(x, y)]))[0][0]

                # NoData 제외
                if src.nodata is not None and val == src.nodata:
                    return None, str(dem_file)

                return float(val), str(dem_file)

    return None, None

In [10]:
elev, used_file = find_elevation_from_tiles(127.0276, 37.4979, dem_root="DEM")
print("고도:", elev)
print("사용 타일:", used_file)

고도: 17.626575469970703
사용 타일: DEM\37705\37705.img


In [12]:
from pathlib import Path
import rasterio
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)

def find_elevation_from_tiles(lon, lat, dem_root="DEM"):
    x, y = transformer.transform(lon, lat)
    dem_files = list(Path(dem_root).rglob("*.img"))

    print("변환 좌표:", x, y)
    print("찾은 타일 수:", len(dem_files))

    for dem_file in dem_files:
        with rasterio.open(dem_file) as src:
            b = src.bounds

            if b.left <= x <= b.right and b.bottom <= y <= b.top:
                val = list(src.sample([(x, y)]))[0][0]
                print("사용 타일:", dem_file)
                print("nodata:", src.nodata)
                print("고도값:", val)
                return float(val) if val != src.nodata else None

    print("해당 좌표를 포함하는 타일 없음")
    return None

elev = find_elevation_from_tiles(127.0276, 37.4979, dem_root="DEM")
print("최종 고도:", elev)

변환 좌표: 958243.2359761745 1944398.131422904
찾은 타일 수: 21
사용 타일: DEM\37705\37705.img
nodata: -9999.0
고도값: 17.626575
최종 고도: 17.626575469970703


In [1]:
import pandas as pd
df = pd.read_excel('놀이친구/[친구] 0~18세 인구통계.xlsx')
df

,시도명,시군구명,읍면동명,남자,여자,0세남자,1세남자,2세남자,3세남자,4세남자,...,9세여자,10세여자,11세여자,12세여자,13세여자,14세여자,15세여자,16세여자,17세여자,18세여자
0,서울특별시,종로구,청운효자동,4895,5919,17,21,21,20,20,...,37,40,38,47,55,44,48,43,56,58
1,서울특별시,종로구,사직동,3921,4986,6,19,11,21,22,...,33,23,35,32,40,35,29,25,35,40
2,서울특별시,종로구,삼청동,996,1103,5,3,1,3,4,...,2,8,4,12,6,10,5,6,9,7
3,서울특별시,종로구,부암동,4154,4750,15,16,14,15,12,...,27,31,34,36,40,28,29,27,30,44
4,서울특별시,종로구,평창동,7827,9147,36,37,42,26,51,...,46,68,58,70,79,70,86,66,87,86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
858,경기도,김포시,풍무동,28853,30819,168,173,195,213,231,...,307,327,350,335,382,368,372,321,304,321
859,경기도,김포시,장기동,19695,20571,106,98,131,139,178,...,289,318,285,291,333,295,310,290,287,297
860,경기도,김포시,구래동,22797,22833,169,165,168,196,177,...,309,320,353,281,319,314,257,278,199,217
861,경기도,김포시,마산동,17458,17999,153,153,149,184,207,...,286,304,284,245,263,260,207,195,179,136


In [2]:
df['읍면동명'] = df['읍면동명'].str.replace(r'제(?=\d)', '', n=1, regex=True)

In [4]:
df[df['읍면동명'].str.contains('홍제')]

,시도명,시군구명,읍면동명,남자,여자,0세남자,1세남자,2세남자,3세남자,4세남자,...,9세여자,10세여자,11세여자,12세여자,13세여자,14세여자,15세여자,16세여자,17세여자,18세여자
198,서울특별시,서대문구,홍제1동,11280,13006,70,62,54,64,42,...,70,88,83,73,78,72,92,90,74,96
199,서울특별시,서대문구,홍제3동,8199,9077,40,52,42,33,52,...,45,42,60,43,49,39,43,57,52,78
200,서울특별시,서대문구,홍제2동,5532,6379,37,31,39,26,32,...,30,24,33,37,34,35,31,35,49,44


In [5]:
df.to_excel('놀이친구/[친구] 0~18세 인구통계.xlsx')

In [12]:
a = pd.read_excel('[최종]진짜마지막최종_final.xlsx')
a

,Unnamed: 0,시군구,행정동,복지_지수,안전_지수,교육_지수,환경_지수,건강_지수,총점,100점 환산
0,0,연수구,송도2동,0.000993,0.261407,0.053170,0.124219,0.109911,0.549702,84.378765
1,1,연수구,송도1동,0.000510,0.261407,0.060113,0.087210,0.093345,0.502585,77.127319
2,2,동구,송림2동,0.080400,0.177781,0.090221,0.083398,0.088086,0.519885,80.035632
3,3,서구,가좌2동,0.004758,0.250350,0.036773,0.055803,0.096500,0.444184,65.283077
4,4,연수구,동춘2동,0.000564,0.261407,0.047372,0.072056,0.108181,0.489580,74.752663
...,...,...,...,...,...,...,...,...,...,...
858,858,종로구,무악동,0.004044,0.071460,0.033293,0.068713,0.004231,0.181741,11.994865
859,859,종로구,창신3동,0.009584,0.071460,0.027326,0.068661,0.000000,0.177031,11.506469
860,860,종로구,숭인1동,0.000000,0.071460,0.005348,0.069593,0.000000,0.146400,8.735790
861,861,종로구,평창동,0.003279,0.071460,0.001072,0.068804,0.000000,0.144615,8.594515


In [9]:
a['시군구'] = a['시군구'].str.replace(' ', '')

In [13]:
a[a['행정동'].str.contains('상일')]

,Unnamed: 0,시군구,행정동,복지_지수,안전_지수,교육_지수,환경_지수,건강_지수,총점,100점 환산
157,157,강동구,상일1동,0.000477,0.216406,0.044898,0.070608,0.092106,0.424494,60.697268
213,213,강동구,상일2동,0.001624,0.216406,0.041066,0.068753,0.083616,0.411465,57.549726


In [11]:
a.to_excel('[최종]진짜마지막최종_final.xlsx')

In [14]:
aprtment = pd.read_excel('house/인천서울경기_단독_다가구_매매.xlsx')
aprtment

,도로명,평균금액,시군구,주택유형,위도,경도
0,가곡로,2.900000e+08,경기도 남양주시 화도읍 가곡리,단독,37.688420,127.299375
1,가곡로88번길,9.406300e+08,경기도 남양주시 화도읍 가곡리,단독,37.692982,127.301506
2,가운로,4.700000e+08,경기도 남양주시 다산동,단독,37.599058,127.161750
3,가운로3길,1.150000e+09,경기도 남양주시 다산동,단독,37.597459,127.158032
4,가현로105번길,1.200000e+08,경기도 김포시 통진읍 가현리,단독,37.673288,126.580836
...,...,...,...,...,...,...
5114,석산로208번길,5.779500e+08,인천광역시 남동구 구월동,단독,37.459010,126.715312
5115,석산로208번길,5.137000e+08,인천광역시 남동구 구월동,단독,37.459010,126.715312
5116,대청로7번안길,8.000000e+07,인천광역시 옹진군 대청면 대청리,다가구,37.825831,124.713716
5117,참외전로244번길,1.300000e+08,인천광역시 중구 도원동,다가구,37.467671,126.638845


In [21]:
aprtment.describe()

,평균금액,위도,경도
count,5.118000e+03,5119.000000,5119.000000
mean,1.233863e+09,37.529634,126.920752
std,1.655431e+09,0.111077,0.247656
min,1.000000e+07,36.992792,124.637509
25%,4.296250e+08,37.478551,126.811225
50%,8.000000e+08,37.537682,126.973218
75%,1.410000e+09,37.589729,127.073970
max,3.738600e+10,38.040942,127.629725


In [1]:
import pandas as pd
df_s = pd.read_excel('놀이친구/[친구] 0~18세 인구통계.xlsx')
df_s

,Unnamed: 0,시도명,시군구명,읍면동명,남자,여자,0세남자,1세남자,2세남자,3세남자,...,9세여자,10세여자,11세여자,12세여자,13세여자,14세여자,15세여자,16세여자,17세여자,18세여자
0,0,서울특별시,종로구,청운효자동,4895,5919,17,21,21,20,...,37,40,38,47,55,44,48,43,56,58
1,1,서울특별시,종로구,사직동,3921,4986,6,19,11,21,...,33,23,35,32,40,35,29,25,35,40
2,2,서울특별시,종로구,삼청동,996,1103,5,3,1,3,...,2,8,4,12,6,10,5,6,9,7
3,3,서울특별시,종로구,부암동,4154,4750,15,16,14,15,...,27,31,34,36,40,28,29,27,30,44
4,4,서울특별시,종로구,평창동,7827,9147,36,37,42,26,...,46,68,58,70,79,70,86,66,87,86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
858,858,경기도,김포시,풍무동,28853,30819,168,173,195,213,...,307,327,350,335,382,368,372,321,304,321
859,859,경기도,김포시,장기동,19695,20571,106,98,131,139,...,289,318,285,291,333,295,310,290,287,297
860,860,경기도,김포시,구래동,22797,22833,169,165,168,196,...,309,320,353,281,319,314,257,278,199,217
861,861,경기도,김포시,마산동,17458,17999,153,153,149,184,...,286,304,284,245,263,260,207,195,179,136


In [3]:
df_s['시군구명']=df_s['시군구명'].str.replace(' ' ,'', regex=False)

In [7]:
df_s = df_s.iloc[:,1:]

In [8]:
df_s.to_excel('놀이친구/[친구] 0~18세 인구통계.xlsx', index='False')